# Collect ~1000 Theoretical Papers from arXiv (cs.CC · cs.LO · cs.FL)

This notebook queries arXiv for three core CS theory categories, applies a lightweight theory-keyword filter, removes duplicates, and saves outputs to CSV and JSON.

Target categories: `cs.CC` (Computational Complexity), `cs.LO` (Logic in CS), `cs.FL` (Formal Languages & Automata).

Rate-limit rules followed: 100 results per request, 3-second delay between requests.

In [1]:
# %pip install requests pandas
import re
import time
from pathlib import Path
import xml.etree.ElementTree as ET

import pandas as pd
import requests

BASE_URL = "http://export.arxiv.org/api/query"
CATEGORIES = ["cs.CC", "cs.LO", "cs.FL"]
SEARCH_QUERY = " OR ".join(f"cat:{cat}" for cat in CATEGORIES)

TARGET_PAPERS = 1000
MAX_RESULTS_PER_CALL = 100
MAX_REQUESTS = 50
REQUEST_DELAY_SECONDS = 3
REQUEST_TIMEOUT_SECONDS = 60
MIN_KEYWORD_HITS = 1

THEORY_KEYWORDS = [
    "theorem",
    "proof",
    "lemma",
    "corollary",
    "proposition",
    "conjecture",
    "bound",
    "complexity",
    "lower bound",
    "upper bound",
    "asymptotic",
    "approximation ratio",
    "np-hard",
    "np complete",
    "sample complexity",
    "regret bound",
    "convergence rate",
    "combinatorial",
]

OUTPUT_DIR = Path("Dataset collection")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "theory_papers_1000.csv"
OUTPUT_JSON = OUTPUT_DIR / "theory_papers_1000.json"

ATOM_NS = "{http://www.w3.org/2005/Atom}"
ARXIV_NS = "{http://arxiv.org/schemas/atom}"

session = requests.Session()
session.headers.update({
    "User-Agent": "ResearchPaperCategorizer/1.0 (mailto:replace-with-your-email@example.com)"
})

In [2]:
def clean_text(value: str) -> str:
    return " ".join((value or "").split())


THEORY_PATTERNS = [re.compile(rf"\b{re.escape(keyword)}\b") for keyword in THEORY_KEYWORDS]


def keyword_hits(text: str) -> list[str]:
    lowered = text.lower()
    return [keyword for keyword, pattern in zip(THEORY_KEYWORDS, THEORY_PATTERNS) if pattern.search(lowered)]


def parse_entry(entry) -> dict:
    title = clean_text(entry.findtext(f"{ATOM_NS}title", default=""))
    abstract = clean_text(entry.findtext(f"{ATOM_NS}summary", default=""))
    arxiv_url = clean_text(entry.findtext(f"{ATOM_NS}id", default=""))
    published = clean_text(entry.findtext(f"{ATOM_NS}published", default=""))

    cat_terms = []
    for cat in entry.findall(f"{ATOM_NS}category"):
        term = clean_text(cat.attrib.get("term", ""))
        if term:
            cat_terms.append(term)

    primary_node = entry.find(f"{ARXIV_NS}primary_category")
    primary_category = ""
    if primary_node is not None:
        primary_category = clean_text(primary_node.attrib.get("term", ""))
    if not primary_category and cat_terms:
        primary_category = cat_terms[0]

    arxiv_id = arxiv_url.rsplit("/", 1)[-1] if arxiv_url else ""

    return {
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "published": published,
        "primary_category": primary_category,
        "categories": " ".join(sorted(set(cat_terms))),
        "arxiv_url": arxiv_url,
    }


def fetch_papers(start: int, max_results: int = 100) -> list[dict]:
    params = {
        "search_query": SEARCH_QUERY,
        "start": start,
        "max_results": max_results,
        "sortBy": "submittedDate",
        "sortOrder": "descending",
    }
    response = session.get(BASE_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
    response.raise_for_status()

    root = ET.fromstring(response.text)
    entries = root.findall(f"{ATOM_NS}entry")
    return [parse_entry(entry) for entry in entries]


def is_theory_like(paper: dict, min_hits: int = 1) -> bool:
    text = f"{paper['title']} {paper['abstract']}"
    hits = keyword_hits(text)
    paper["keyword_hits"] = ", ".join(hits)
    paper["keyword_hit_count"] = len(hits)
    return len(hits) >= min_hits

In [3]:
theory_papers = []
seen_ids = set()
seen_titles = set()

start = 0
for request_idx in range(1, MAX_REQUESTS + 1):
    batch = fetch_papers(start=start, max_results=MAX_RESULTS_PER_CALL)
    if not batch:
        print("No more results returned by arXiv.")
        break

    accepted = 0
    for paper in batch:
        if not paper["title"] or not paper["abstract"]:
            continue

        title_key = re.sub(r"\s+", " ", paper["title"].lower()).strip()
        if paper["arxiv_id"] in seen_ids or title_key in seen_titles:
            continue

        if is_theory_like(paper, min_hits=MIN_KEYWORD_HITS):
            seen_ids.add(paper["arxiv_id"])
            seen_titles.add(title_key)
            theory_papers.append(paper)
            accepted += 1

        if len(theory_papers) >= TARGET_PAPERS:
            break

    print(
        f"Request {request_idx}/{MAX_REQUESTS} | start={start} | fetched={len(batch)} | accepted={accepted} | total={len(theory_papers)}"
    )

    if len(theory_papers) >= TARGET_PAPERS:
        break

    start += MAX_RESULTS_PER_CALL
    time.sleep(REQUEST_DELAY_SECONDS)

print(f"Collected {len(theory_papers)} theory-like papers.")

Request 1/50 | start=0 | fetched=100 | accepted=57 | total=57
Request 2/50 | start=100 | fetched=100 | accepted=55 | total=112
Request 3/50 | start=200 | fetched=100 | accepted=58 | total=170
Request 4/50 | start=300 | fetched=100 | accepted=60 | total=230
Request 5/50 | start=400 | fetched=100 | accepted=69 | total=299
Request 6/50 | start=500 | fetched=100 | accepted=58 | total=357
Request 7/50 | start=600 | fetched=100 | accepted=47 | total=404
Request 8/50 | start=700 | fetched=100 | accepted=50 | total=454
Request 9/50 | start=800 | fetched=100 | accepted=50 | total=504
Request 10/50 | start=900 | fetched=100 | accepted=47 | total=551
Request 11/50 | start=1000 | fetched=100 | accepted=49 | total=600
Request 12/50 | start=1100 | fetched=100 | accepted=71 | total=671
Request 13/50 | start=1200 | fetched=100 | accepted=53 | total=724
Request 14/50 | start=1300 | fetched=100 | accepted=46 | total=770
Request 15/50 | start=1400 | fetched=100 | accepted=71 | total=841
Request 16/50 | s

In [4]:
df = pd.DataFrame(theory_papers)
if df.empty:
    raise RuntimeError("No papers collected. Try reducing MIN_KEYWORD_HITS or increasing MAX_REQUESTS.")

df = df.head(TARGET_PAPERS).copy()
df.insert(0, "label", "THEORETICAL")
df.insert(1, "source_query", SEARCH_QUERY)

# Keep a project-friendly column order.
ordered_cols = [
    "label",
    "title",
    "abstract",
    "primary_category",
    "categories",
    "keyword_hit_count",
    "keyword_hits",
    "published",
    "arxiv_id",
    "arxiv_url",
    "source_query",
]
df = df[ordered_cols]

df.to_csv(OUTPUT_CSV, index=False)
df.to_json(OUTPUT_JSON, orient="records", indent=2, force_ascii=False)

print(f"Saved {len(df):,} rows to {OUTPUT_CSV}")
print(f"Saved {len(df):,} rows to {OUTPUT_JSON}")
print()
print("Top primary categories:")
print(df["primary_category"].value_counts().head(10))
print()
print("Keyword hit distribution:")
print(df["keyword_hit_count"].value_counts().sort_index())

df.head(5)

Saved 1,000 rows to Dataset collection/theory_papers_1000.csv
Saved 1,000 rows to Dataset collection/theory_papers_1000.json

Top primary categories:
primary_category
cs.LO       278
cs.CC       232
quant-ph     91
cs.FL        64
cs.DS        61
cs.AI        34
cs.LG        33
math.CO      29
math.LO      23
cs.GT        18
Name: count, dtype: int64

Keyword hit distribution:
keyword_hit_count
1    557
2    265
3    117
4     45
5     11
6      5
Name: count, dtype: int64


,label,title,abstract,primary_category,categories,keyword_hit_count,keyword_hits,published,arxiv_id,arxiv_url,source_query
0,THEORETICAL,Smaller Depth-2 Linear Circuits for Disjointne...,We prove two new upper bounds for depth-2 line...,cs.CC,cs.CC,2,"bound, upper bound",2026-03-16T17:26:33Z,2603.15565v1,http://arxiv.org/abs/2603.15565v1,cat:cs.CC OR cat:cs.LO OR cat:cs.FL
1,THEORETICAL,Completeness of Relational Algebra via Cylindr...,An alternative proof of the completeness of re...,cs.LO,cs.DB cs.LO math.LO,1,proof,2026-03-16T10:50:45Z,2603.15099v1,http://arxiv.org/abs/2603.15099v1,cat:cs.CC OR cat:cs.LO OR cat:cs.FL
2,THEORETICAL,Convex algebras on an interval with semicontin...,In a recent work of Matteo Mio on compact quan...,cs.LO,cs.LO math.LO,1,theorem,2026-03-16T08:10:05Z,2603.14955v1,http://arxiv.org/abs/2603.14955v1,cat:cs.CC OR cat:cs.LO OR cat:cs.FL
3,THEORETICAL,Towards Parameterized Hardness on Maintaining ...,We investigate the fine-grained complexity of ...,cs.DB,cs.CC cs.DB,4,"bound, complexity, lower bound, upper bound",2026-03-16T02:39:08Z,2603.14754v1,http://arxiv.org/abs/2603.14754v1,cat:cs.CC OR cat:cs.LO OR cat:cs.FL
4,THEORETICAL,Decision Quotient: A Regime-Sensitive Complexi...,Which coordinates of a decision problem can be...,cs.CC,cs.CC,1,complexity,2026-03-16T00:48:15Z,2603.14689v1,http://arxiv.org/abs/2603.14689v1,cat:cs.CC OR cat:cs.LO OR cat:cs.FL


In [5]:
# Quick manual spot-check (10 examples)
preview_cols = ["title", "primary_category", "keyword_hits"]
df.sample(n=min(10, len(df)), random_state=42)[preview_cols].reset_index(drop=True)

,title,primary_category,keyword_hits
0,Sampling Permutations with Cell Probes is Hard,cs.CC,"bound, lower bound"
1,The Parameterized Complexity of Computing the ...,cs.CC,complexity
2,Near-Optimal Quantum Algorithms for Computing ...,quant-ph,complexity
3,Efficient Testing Implies Structured Symmetry,cs.CC,complexity
4,Stronger Approximation Guarantees for Non-Mono...,cs.LG,bound
5,Scheduling Problems with Constrained Rejections,cs.DS,np-hard
6,Word equations and the exponent of periodicity,cs.FL,conjecture
7,New Perspectives on Semiring Applications to D...,cs.CC,"complexity, np-hard, combinatorial"
8,Transporting Theorems about Typeability in LF ...,cs.LO,"proof, bound"
9,Nazrin: Atomic Tactics for Graph Neural Networ...,cs.LO,"theorem, proof, conjecture"
